# Volatility index curves (e.g. VIX term structure)

Deep-dive reference: **forward levels of a volatility index** across horizon — not equity option implied vol at a single strike.

## Concept

Indices like **VIX** quote **implied volatility** of a model portfolio of options; the **term structure** is how that index level is expected to evolve by horizon. **`VolatilityIndexCurve`** stores **`(t, forward_level)`** knots where the level is in **vol points** (e.g. 18 for 18% vol index reading), depending on convention.

## API walkthrough

Use **`forward_level(t)`** to read the curve at year fraction `t`.

In [ ]:
try:
    from datetime import date

    from finstack.core.market_data import VolatilityIndexCurve
except ImportError as exc:
    print("VolatilityIndexCurve is not importable:", exc)
else:
    base = date(2024, 1, 1)
    vix = VolatilityIndexCurve(
        "VIX",
        base,
        [
            (0.0, 16.0),
            (1 / 12, 17.5),
            (0.25, 19.0),
            (0.5, 20.0),
            (1.0, 22.0),
        ],
        day_count="act_365f",
    )
    print("curve:", vix)
    print("Term structure (implied index forward level, stylized vol points):")
    for t in (0.0, 1 / 12, 0.25, 0.5, 1.0):
        print(f"  t={t:.4f}y -> forward_level = {vix.forward_level(t):.4f}")

## Practical example

Compare **front** vs **1Y** forward index level — often used in **vol-of-vol** or **dispersion** discussions at a high level.

In [ ]:
try:
    from datetime import date

    from finstack.core.market_data import VolatilityIndexCurve
except ImportError as exc:
    print("Skip example — VolatilityIndexCurve unavailable:", exc)
else:
    base = date(2024, 1, 1)
    vix = VolatilityIndexCurve("VIX", base, [(0.0, 18.0), (1.0, 21.0)], day_count="act_365f")
    front = vix.forward_level(0.0)
    one_y = vix.forward_level(1.0)
    print(f"Front forward level: {front:.2f}")
    print(f"1Y forward level:    {one_y:.2f}")
    print(f"Spread (1Y - spot curve front): {one_y - front:.2f} vol points")

## Takeaways

- This is a **1D term structure of index levels**, not a **strike / expiry vol surface** for one underlying.
- **`forward_level(t)`** interpolates the **stylized** market view of future index values.
- For **options on a single name**, you typically need a **`VolSurface`** (when exposed in your stack) or instrument-specific vol inputs.